In [5]:
import pandas as pd
import statsmodels.api as sm
from itertools import combinations
from statsmodels.tsa.stattools import adfuller



In [ ]:
#finding beta
def hedge_ratio(y: pd.Series, x: pd.Series ) -> float:
    model = sm.OLS(y, sm.add_constant(x)).fit()
    return model.params.iloc[1]  # Return the slope  from the model (Beta)

In [ ]:
#ADF test for cointegration and flatness
def engle_granger_pvalue(y: pd.Series, x: pd.Series) -> float:
    df = pd.concat([y, x], axis=1).dropna()
    y_aligned, x_aligned = df.iloc[:, 0], df.iloc[:, 1]
    #cleaned and aligned

    beta = hedge_ratio(y_aligned, x_aligned)
    spread = y_aligned - beta * x_aligned

    return float(adfuller(spread, regression="c", autolag="AIC")[1])

In [6]:
def find_cointegrated_pairs(prices: pd.DataFrame, max_pvalue: float = 0.05) -> pd.DataFrame:
    rows = []

    for a, b in combinations(prices.columns, 2):
        y, x = prices[a], prices[b]
        rows.append({
            "a": a,
            "b": b,
            "pvalue": engle_granger_pvalue(y, x),
            "beta": hedge_ratio(y, x),
        })

    result = pd.DataFrame(rows, columns=["a", "b", "pvalue", "beta"])
    result = result[result["pvalue"] <= max_pvalue]
    return result.sort_values("pvalue").reset_index(drop=True)